# M5b · Backfill `fact_telemetry_daily`

**Goal:** aggregate `staging.archive` pings into per-(tenant, device, date)
rows. Compresses the 55M-row archive into a few-hundred-thousand-row table
suitable for monthly mart aggregation.

**SQL file:** `sql/16_fact_telemetry_daily_incr.sql`.
Bound parameters: `:window_start`, `:window_end`, `:etl_run_id`,
`:ping_seconds` (default 30), `:rpm_high_threshold` (default 3000).

**What lands per row:** observation_count, ignition_on_minutes,
moving_minutes, idle_minutes, idle_ratio, avg/max speed,
avg/max RPM, high_rpm_seconds, total_fuel_used.

**Exit criteria:**
- One row per active device per day (modulo data gaps).
- `idle_ratio` distributed in [0, 1] — no NaNs after COALESCE.
- Watermark advances to last archive day.

> **Run order:** can run in parallel with `06_load_fact_harsh_event` —
> the two facts read the same staging table but write to different fact
> tables and never contend on the same rows.


In [1]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


DB = medali_dev@localhost:15432/accent_data
Schemas: staging=staging, warehouse=warehouse, marts=marts


## 2. Inputs

In [2]:
from accent_fleet.config import load_pipeline_config
from accent_fleet.db.sql_loader import load_sql
from datetime import datetime, timedelta

cfg = load_pipeline_config()
CHUNK_DAYS = cfg['window']['backfill_chunk_days']
MIN_TIME   = datetime.fromisoformat(cfg['window']['min_event_time'].replace('Z','+00:00')).replace(tzinfo=None)
TEL = cfg.get('archive_telemetry', {})
print('chunk_days =', CHUNK_DAYS, '  min_time =', MIN_TIME)
print('telemetry params =', TEL)

with get_engine().connect() as conn:
    end_time = conn.execute(text("SELECT MAX(date) FROM staging.archive")).scalar_one()
print('staging.archive max event-time =', end_time)

print('\n----- SQL file preview -----')
print(load_sql('16_fact_telemetry_daily_incr.sql')[:800], '...')


chunk_days = 30   min_time = 2019-10-01 00:00:00
telemetry params = {'ping_seconds': 30, 'rpm_high_threshold': 3000}
staging.archive max event-time = 2026-04-10 14:11:03

----- SQL file preview -----
-- =============================================================================
-- 16_fact_telemetry_daily_incr.sql
-- =============================================================================
-- fact_telemetry_daily: per-(tenant, device, date) aggregate of staging.archive
-- ping data. Compresses ~10-100k pings/day/device into a single row carrying
-- the signals needed for monthly behaviour features:
--
--   - observation_count         pings recorded (volume / data-quality proxy)
--   - ignition_on_minutes        approximate engine-on duration
--   - moving_minutes             pings with ignition=1 AND speed>0
--   - idle_minutes               pings with ignition=1 AND speed=0
--   - avg_speed_kmh / max_speed_kmh
--   - avg_rpm / max_rpm
--   - high_rpm_seconds           pings with 

## 3. Execute

In [3]:
import importlib, time, pandas as pd
import accent_fleet.transforms.facts as facts_module
from accent_fleet.pipeline.run_log import begin_run, end_run

facts_module = importlib.reload(facts_module)
load_fact_incremental = facts_module.load_fact_incremental

# Daily aggregation is heavy on GROUP BY — keep chunks small.
TEL_CHUNK_DAYS = 7

run_id = begin_run(mode='notebook:07_load_fact_telemetry_daily')
print('etl_run_id =', run_id)

progress = []
cursor = MIN_TIME
t0 = time.time()
try:
    while cursor < end_time:
        next_cursor = min(cursor + timedelta(days=TEL_CHUNK_DAYS), end_time)
        res = load_fact_incremental('fact_telemetry_daily', run_id=run_id, window_end=next_cursor)
        progress.append({'window_start': cursor, 'window_end': next_cursor,
                         'rows_loaded': res.rows_loaded})
        cursor = next_cursor
    end_run(run_id, status='success', rows_loaded=sum(p['rows_loaded'] for p in progress))
except Exception as exc:
    end_run(run_id, status='failed', error_message=str(exc))
    raise

print(f'done in {time.time()-t0:.1f}s — {len(progress)} chunks')
pd.DataFrame(progress).tail(10)


etl_run_id = 29
2026-04-29 20:57:57 [info     ] fact_load.start                end=datetime.datetime(2019, 10, 8, 0, 0) fact=fact_telemetry_daily run_id=29 start=datetime.datetime(2019, 10, 1, 0, 0)
2026-04-29 20:57:57 [info     ] fact_load.done                 fact=fact_telemetry_daily new_watermark=datetime.datetime(2019, 10, 8, 0, 0) rows=1
2026-04-29 20:57:58 [info     ] fact_load.start                end=datetime.datetime(2019, 10, 15, 0, 0) fact=fact_telemetry_daily run_id=29 start=datetime.datetime(2019, 10, 7, 23, 50)
2026-04-29 20:57:58 [info     ] fact_load.done                 fact=fact_telemetry_daily new_watermark=datetime.datetime(2019, 10, 15, 0, 0) rows=0
2026-04-29 20:57:59 [info     ] fact_load.start                end=datetime.datetime(2019, 10, 22, 0, 0) fact=fact_telemetry_daily run_id=29 start=datetime.datetime(2019, 10, 14, 23, 50)
2026-04-29 20:57:59 [info     ] fact_load.done                 fact=fact_telemetry_daily new_watermark=datetime.datetime(2019, 10, 22

,window_start,window_end,rows_loaded
331,2026-02-03,2026-02-10 00:00:00,2445
332,2026-02-10,2026-02-17 00:00:00,2446
333,2026-02-17,2026-02-24 00:00:00,2429
334,2026-02-24,2026-03-03 00:00:00,2439
335,2026-03-03,2026-03-10 00:00:00,2420
336,2026-03-10,2026-03-17 00:00:00,2442
337,2026-03-17,2026-03-24 00:00:00,2217
338,2026-03-24,2026-03-31 00:00:00,2467
339,2026-03-31,2026-04-07 00:00:00,2448
340,2026-04-07,2026-04-10 14:11:03,1442


## 4. Inspect

In [4]:
import pandas as pd
with get_engine().connect() as conn:
    wm = pd.read_sql(text("SELECT * FROM warehouse.etl_watermark WHERE table_name='fact_telemetry_daily'"), conn)
    total = conn.execute(text('SELECT COUNT(*) FROM warehouse.fact_telemetry_daily')).scalar_one()
    distros = pd.read_sql(text('''
        SELECT
          PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY idle_ratio)         AS p50_idle,
          PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY idle_ratio)         AS p95_idle,
          AVG(observation_count)                                            AS avg_obs_per_day,
          AVG(ignition_on_minutes)                                          AS avg_ignition_min,
          AVG(moving_minutes)                                               AS avg_moving_min
        FROM warehouse.fact_telemetry_daily
        WHERE idle_ratio IS NOT NULL
    '''), conn)
    sample = pd.read_sql(text('''
        SELECT * FROM warehouse.fact_telemetry_daily
        ORDER BY telemetry_date DESC LIMIT 5
    '''), conn)

print(f'fact_telemetry_daily total rows: {total:,}')
display(wm); display(distros); display(sample)
assert wm['last_event_time'].iloc[0] is not None, 'watermark did not advance'
print('OK — fact_telemetry_daily backfill complete.')


fact_telemetry_daily total rows: 62,553


,layer,table_name,last_event_time,last_run_at,last_etl_run_id,rows_loaded_total,notes
0,warehouse,fact_telemetry_daily,2026-04-10 14:11:03,2026-04-29 22:22:00.335575+00:00,29,64924,Incremental on date::DATE (staging.archive); p...


,p50_idle,p95_idle,avg_obs_per_day,avg_ignition_min,avg_moving_min
0,0.068998,0.63807,998.247923,453.15259,415.590574


,telemetry_sk,tenant_id,device_id,telemetry_date,observation_count,ignition_on_minutes,moving_minutes,idle_minutes,avg_speed_kmh,max_speed_kmh,avg_rpm,max_rpm,high_rpm_seconds,total_fuel_used,idle_ratio,_etl_run_id,_loaded_at
0,63486,235,425101,2026-04-10,15,0.0,0.0,0.0,0.000000,0,0.000000,0,0.0,0.000000,NaN,29,2026-04-29 22:22:00.335575+00:00
1,63491,235,425103,2026-04-10,397,12.0,7.5,4.5,24.889169,80,885.088161,1766,0.0,256.483333,0.375000,29,2026-04-29 22:22:00.335575+00:00
2,63495,235,425105,2026-04-10,728,354.0,345.0,9.0,20.219780,56,0.000000,0,0.0,0.000000,0.025424,29,2026-04-29 22:22:00.335575+00:00
3,63499,235,425106,2026-04-10,1701,834.0,805.5,28.5,21.034685,86,1005.339212,1884,0.0,3234.483333,0.034173,29,2026-04-29 22:22:00.335575+00:00
4,63503,235,425110,2026-04-10,747,363.0,353.0,10.0,35.838019,72,0.000000,0,0.0,0.000000,0.027548,29,2026-04-29 22:22:00.335575+00:00


OK — fact_telemetry_daily backfill complete.
